In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    UperNetForSemanticSegmentation,
    AutoImageProcessor,
)
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [2]:
# GPU 환경 확인
print("=" * 60)
print("[환경 확인]")
print(f"  CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU 이름: {torch.cuda.get_device_name()}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM 총량: {vram_total:.1f} GB")
    torch.cuda.empty_cache()
print("=" * 60)

[환경 확인]
  CUDA 사용 가능: True
  GPU 이름: NVIDIA GeForce RTX 4060
  VRAM 총량: 8.0 GB


In [3]:
# 설정값
MODEL_TYPE = "upernet"  # "segformer_b0" / "segformer_b2" / "upernet"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output\upernet"
SAVE_DIR = r"D:\Work\study_full_data\image_segmentation\output\upernet\figures"
NUM_SAMPLES = 6          # 추론 시각화할 이미지 수
RUN_COMPARE = False       # 다중 모델 비교 실행 여부

In [4]:
# 클래스 정의
id2label = {
    0: "unlabeled", 1: "paved-area", 2: "dirt", 3: "grass",
    4: "gravel", 5: "water", 6: "rocks", 7: "pool",
    8: "vegetation", 9: "roof", 10: "wall", 11: "window",
    12: "door", 13: "fence", 14: "fence-pole", 15: "person",
    16: "dog", 17: "car", 18: "bicycle", 19: "tree",
    20: "bald-tree", 21: "ar-marker", 22: "obstacle", 23: "conflicting",
}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)
 
LABEL_COLORMAP = {
    0: (0, 0, 0), 1: (128, 64, 128), 2: (130, 76, 0),
    3: (0, 102, 0), 4: (112, 103, 87), 5: (28, 42, 168),
    6: (48, 41, 30), 7: (0, 50, 89), 8: (107, 142, 35),
    9: (70, 70, 70), 10: (102, 102, 156), 11: (254, 228, 12),
    12: (254, 148, 12), 13: (190, 153, 153), 14: (153, 153, 153),
    15: (255, 22, 96), 16: (102, 51, 0), 17: (9, 143, 150),
    18: (119, 11, 32), 19: (51, 51, 0), 20: (190, 250, 190),
    21: (112, 150, 146), 22: (2, 135, 115), 23: (255, 0, 0),
}
 
TARGET_SIZE = (512, 512)

In [5]:
# 학습 곡선 시각화
def plot_training_curves(log_path, save_dir, model_name):
    with open(log_path, "r") as f:
        logs = json.load(f)
 
    train_loss, eval_loss, eval_miou = [], [], []
    train_epochs, eval_epochs = [], []
 
    for entry in logs:
        if "loss" in entry and "eval_loss" not in entry:
            train_loss.append(entry["loss"])
            train_epochs.append(entry.get("epoch", 0))
        if "eval_loss" in entry:
            eval_loss.append(entry["eval_loss"])
            eval_miou.append(entry.get("eval_mean_iou", 0))
            eval_epochs.append(entry.get("epoch", 0))
 
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Training Curves - {model_name}", fontsize=14, fontweight="bold")
 
    axes[0].plot(train_epochs, train_loss, "b-", alpha=0.7)
    axes[0].set_title("Train Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
 
    axes[1].plot(eval_epochs, eval_loss, "r-o", markersize=4)
    axes[1].set_title("Eval Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)
 
    axes[2].plot(eval_epochs, eval_miou, "g-o", markersize=4)
    axes[2].set_title("Mean IoU")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("mIoU")
    axes[2].grid(True, alpha=0.3)
 
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"training_curves_{model_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")

In [6]:
# 카테고리별 IoU 차트
def plot_per_category_iou(eval_path, save_dir, model_name):
    with open(eval_path, "r") as f:
        results = json.load(f)
 
    categories, ious = [], []
    for i in range(num_labels):
        cat = id2label[i]
        iou = results.get(f"eval_iou_{cat}", 0)
        if not np.isnan(iou) and i != 0:  # unlabeled 제외
            categories.append(cat)
            ious.append(iou)
 
    sorted_pairs = sorted(zip(ious, categories), reverse=True)
    ious, categories = zip(*sorted_pairs)
 
    fig, ax = plt.subplots(figsize=(12, 7))
    colors = plt.cm.RdYlGn(np.array(ious))
    bars = ax.barh(range(len(categories)), ious, color=colors)
    ax.set_yticks(range(len(categories)))
    ax.set_yticklabels(categories)
    ax.set_xlabel("IoU")
    ax.set_title(f"Per-Category IoU - {model_name}", fontsize=14, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.grid(True, axis="x", alpha=0.3)
    ax.invert_yaxis()
 
    for bar, iou in zip(bars, ious):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{iou:.3f}", va="center", fontsize=9)
 
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"per_category_iou_{model_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")

In [7]:
# 예측 맵을 컬러로 변환
def prediction_to_color(pred_map):
    h, w = pred_map.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    for label_id, color in LABEL_COLORMAP.items():
        color_map[pred_map == label_id] = color
    return color_map

In [8]:
# 추론 및 시각화
def inference_and_visualize(model_dir, image_dir, label_dir, save_dir,
                            model_type, num_samples, output_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    if model_type.startswith("segformer"):
        model = SegformerForSemanticSegmentation.from_pretrained(
            model_dir, id2label=id2label, label2id=label2id
        )
        processor = SegformerImageProcessor.from_pretrained(model_dir)
    else:
        model = UperNetForSemanticSegmentation.from_pretrained(
            model_dir, id2label=id2label, label2id=label2id
        )
        processor = AutoImageProcessor.from_pretrained(model_dir)
 
    model = model.to(device)
    model.eval()
 
    # 테스트셋 파일명만 사용 (split_info.json에서 로드)
    split_path = os.path.join(output_dir, "split_info.json")
    if os.path.exists(split_path):
        with open(split_path, "r") as f:
            split_info = json.load(f)
        image_files = sorted(split_info["test_files"])
        print(f"Using {len(image_files)} test-only images from split_info.json")
    else:
        print("WARNING: split_info.json not found, using all images")
        image_files = sorted([
            f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
 
    indices = np.linspace(0, len(image_files) - 1, num_samples, dtype=int)
 
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis, :]
 
    all_preds, all_gts = [], []
 
    for idx, img_idx in enumerate(indices):
        img_file = image_files[img_idx]
        file_name = os.path.splitext(img_file)[0]
 
        image = Image.open(os.path.join(image_dir, img_file)).convert("RGB")
        image_resized = image.resize(TARGET_SIZE, Image.BILINEAR)
 
        label_path = os.path.join(label_dir, f"{file_name}.png")
        gt_label = Image.open(label_path).convert("L").resize(TARGET_SIZE, Image.NEAREST)
        gt_array = np.array(gt_label)
 
        inputs = processor(images=[image_resized], return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            upsampled = nn.functional.interpolate(
                logits, size=TARGET_SIZE, mode="bilinear", align_corners=False
            )
            pred = upsampled.argmax(dim=1).squeeze().cpu().numpy()
 
        all_preds.append(pred.flatten())
        all_gts.append(gt_array.flatten())
 
        axes[idx, 0].imshow(image_resized)
        axes[idx, 0].set_title(f"Original: {img_file}", fontsize=10)
        axes[idx, 0].axis("off")
 
        axes[idx, 1].imshow(prediction_to_color(pred))
        axes[idx, 1].set_title("Predicted Segmentation", fontsize=10)
        axes[idx, 1].axis("off")
 
        axes[idx, 2].imshow(prediction_to_color(gt_array))
        axes[idx, 2].set_title("Ground Truth", fontsize=10)
        axes[idx, 2].axis("off")
 
    plt.suptitle(f"Inference Results - {model_type}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"inference_{model_type}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")
 
    save_legend(save_dir)
 
    all_preds = np.concatenate(all_preds)
    all_gts = np.concatenate(all_gts)
    plot_confusion_matrix(all_preds, all_gts, save_dir, model_type)

In [9]:
# 혼동 행렬
def plot_confusion_matrix(preds, gts, save_dir, model_name):
    present_classes = sorted(set(np.unique(gts)) | set(np.unique(preds)))
    present_classes = [c for c in present_classes if c < num_labels]
 
    cm = confusion_matrix(gts, preds, labels=present_classes, normalize="true")
    class_names = [id2label.get(c, str(c)) for c in present_classes]
 
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, vmin=0, vmax=1,
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Ground Truth", fontsize=12)
    ax.set_title(f"Normalized Confusion Matrix - {model_name}", fontsize=14, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
 
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"confusion_matrix_{model_name}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")

In [10]:
# 컬러맵 범례
def save_legend(save_dir):
    fig, ax = plt.subplots(figsize=(4, 8))
    patches = []
    for i in range(num_labels):
        color = np.array(LABEL_COLORMAP[i]) / 255.0
        patches.append(mpatches.Patch(color=color, label=f"{i}: {id2label[i]}"))
 
    ax.legend(handles=patches, loc="center", fontsize=9, frameon=False)
    ax.axis("off")
    ax.set_title("Class Legend", fontsize=12, fontweight="bold")
 
    plt.tight_layout()
    save_path = os.path.join(save_dir, "class_legend.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")

In [18]:
# 다중 모델 비교표
def compare_models(output_dir, save_dir):
    model_types = ["segformer_b0", "segformer_b2", "upernet"]
    results = {}
 
    for mt in model_types:
        eval_path = os.path.join(output_dir, mt, f"eval_{mt}.json")
        if os.path.exists(eval_path):
            with open(eval_path, "r") as f:
                results[mt] = json.load(f)
 
    if len(results) < 2:
        print("Need at least 2 model results for comparison. Skipping.")
        return
 
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
    model_names = list(results.keys())
    metrics_keys = ["eval_mean_iou", "eval_mean_accuracy", "eval_overall_accuracy"]
    metrics_labels = ["Mean IoU", "Mean Accuracy", "Overall Accuracy"]
 
    x = np.arange(len(metrics_labels))
    width = 0.25
 
    for i, mn in enumerate(model_names):
        vals = [results[mn].get(k, 0) for k in metrics_keys]
        axes[0].bar(x + i * width, vals, width, label=mn)
 
    axes[0].set_xticks(x + width * (len(model_names) - 1) / 2)
    axes[0].set_xticklabels(metrics_labels)
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Model Comparison - Key Metrics", fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, axis="y", alpha=0.3)
 
    first_model = model_names[0]
    cat_ious = {}
    for i in range(1, num_labels):
        cat = id2label[i]
        iou = results[first_model].get(f"eval_iou_{cat}", 0)
        if not np.isnan(iou):
            cat_ious[cat] = iou
 
    top_cats = sorted(cat_ious.keys(), key=lambda c: cat_ious[c], reverse=True)[:10]
 
    x2 = np.arange(len(top_cats))
    for i, mn in enumerate(model_names):
        vals = [results[mn].get(f"eval_iou_{c}", 0) for c in top_cats]
        axes[1].bar(x2 + i * width, vals, width, label=mn)
 
    axes[1].set_xticks(x2 + width * (len(model_names) - 1) / 2)
    axes[1].set_xticklabels(top_cats, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Per-Category IoU (Top 10)", fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, axis="y", alpha=0.3)
 
    plt.tight_layout()
    save_path = os.path.join(save_dir, "model_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")
 
    # 텍스트 비교표
    print(f"\n{'='*70}")
    print(f"  MODEL COMPARISON SUMMARY")
    print(f"{'='*70}")
    print(f"  {'Metric':<25}", end="")
    for mn in model_names:
        print(f"{mn:>15}", end="")
    print()
    print(f"  {'-'*70}")
    for key, label in zip(metrics_keys, metrics_labels):
        print(f"  {label:<25}", end="")
        for mn in model_names:
            val = results[mn].get(key, 0)
            print(f"{val:>15.4f}", end="")
        print()

In [12]:
# 실행
if __name__ == "__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)
 
    model_dir = os.path.join(OUTPUT_DIR, f"model_{MODEL_TYPE}")
 
    # 학습 곡선
    log_path = os.path.join(OUTPUT_DIR, f"log_{MODEL_TYPE}.json")
    if os.path.exists(log_path):
        plot_training_curves(log_path, SAVE_DIR, MODEL_TYPE)
 
    # 카테고리별 IoU
    eval_path = os.path.join(OUTPUT_DIR, f"eval_{MODEL_TYPE}.json")
    if os.path.exists(eval_path):
        plot_per_category_iou(eval_path, SAVE_DIR, MODEL_TYPE)
 
    # 추론 시각화 + 혼동 행렬
    if os.path.exists(model_dir):
        inference_and_visualize(
            model_dir, IMAGE_DIR, LABEL_DIR,
            SAVE_DIR, MODEL_TYPE, NUM_SAMPLES, OUTPUT_DIR
        )
    else:
        print(f"Model not found: {model_dir}")
 
    # 다중 모델 비교
    if RUN_COMPARE:
        compare_models(OUTPUT_DIR, SAVE_DIR)

Saved: D:\Work\study_full_data\image_segmentation\output\upernet\figures\training_curves_upernet.png
Saved: D:\Work\study_full_data\image_segmentation\output\upernet\figures\per_category_iou_upernet.png


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Using 40 test-only images from split_info.json
Saved: D:\Work\study_full_data\image_segmentation\output\upernet\figures\inference_upernet.png
Saved: D:\Work\study_full_data\image_segmentation\output\upernet\figures\class_legend.png
Saved: D:\Work\study_full_data\image_segmentation\output\upernet\figures\confusion_matrix_upernet.png


In [13]:
MODEL_TYPE = "segformer_b0"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b0"
SAVE_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures"
NUM_SAMPLES = 6          # 추론 시각화할 이미지 수
RUN_COMPARE = False       # 다중 모델 비교 실행 여부

if __name__ == "__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)
 
    model_dir = os.path.join(OUTPUT_DIR, f"model_{MODEL_TYPE}")
 
    # 학습 곡선
    log_path = os.path.join(OUTPUT_DIR, f"log_{MODEL_TYPE}.json")
    if os.path.exists(log_path):
        plot_training_curves(log_path, SAVE_DIR, MODEL_TYPE)
 
    # 카테고리별 IoU
    eval_path = os.path.join(OUTPUT_DIR, f"eval_{MODEL_TYPE}.json")
    if os.path.exists(eval_path):
        plot_per_category_iou(eval_path, SAVE_DIR, MODEL_TYPE)
 
    # 추론 시각화 + 혼동 행렬
    if os.path.exists(model_dir):
        inference_and_visualize(
            model_dir, IMAGE_DIR, LABEL_DIR,
            SAVE_DIR, MODEL_TYPE, NUM_SAMPLES, OUTPUT_DIR
        )
    else:
        print(f"Model not found: {model_dir}")
 
    # 다중 모델 비교
    if RUN_COMPARE:
        compare_models(OUTPUT_DIR, SAVE_DIR)

Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures\training_curves_segformer_b0.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures\per_category_iou_segformer_b0.png
Using 40 test-only images from split_info.json
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures\inference_segformer_b0.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures\class_legend.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b0\figures\confusion_matrix_segformer_b0.png


In [14]:
MODEL_TYPE = "segformer_b2"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b2"
SAVE_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures"
NUM_SAMPLES = 6          # 추론 시각화할 이미지 수
RUN_COMPARE = False       # 다중 모델 비교 실행 여부

if __name__ == "__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)
 
    model_dir = os.path.join(OUTPUT_DIR, f"model_{MODEL_TYPE}")
 
    # 학습 곡선
    log_path = os.path.join(OUTPUT_DIR, f"log_{MODEL_TYPE}.json")
    if os.path.exists(log_path):
        plot_training_curves(log_path, SAVE_DIR, MODEL_TYPE)
 
    # 카테고리별 IoU
    eval_path = os.path.join(OUTPUT_DIR, f"eval_{MODEL_TYPE}.json")
    if os.path.exists(eval_path):
        plot_per_category_iou(eval_path, SAVE_DIR, MODEL_TYPE)
 
    # 추론 시각화 + 혼동 행렬
    if os.path.exists(model_dir):
        inference_and_visualize(
            model_dir, IMAGE_DIR, LABEL_DIR,
            SAVE_DIR, MODEL_TYPE, NUM_SAMPLES, OUTPUT_DIR
        )
    else:
        print(f"Model not found: {model_dir}")
 
    # 다중 모델 비교
    if RUN_COMPARE:
        compare_models(OUTPUT_DIR, SAVE_DIR)

Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures\training_curves_segformer_b2.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures\per_category_iou_segformer_b2.png
Using 40 test-only images from split_info.json
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures\inference_segformer_b2.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures\class_legend.png
Saved: D:\Work\study_full_data\image_segmentation\output\segformer_b2\figures\confusion_matrix_segformer_b2.png


In [19]:
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output"
SAVE_DIR = r"D:\Work\study_full_data\image_segmentation\output"
compare_models(OUTPUT_DIR, SAVE_DIR)

Saved: D:\Work\study_full_data\image_segmentation\output\model_comparison.png

  MODEL COMPARISON SUMMARY
  Metric                      segformer_b0   segformer_b2        upernet
  ----------------------------------------------------------------------
  Mean IoU                          0.5015         0.6279         0.6091
  Mean Accuracy                     0.5843         0.7308         0.7361
  Overall Accuracy                  0.8756         0.9048         0.9123
